# Closing Auction — Feature Engineering

Starts from `data/panel_clean.csv`, the cleaned panel built in `01) data_exploration.ipynb`: one row per symbol-day, last pre-close snapshot joined to the realized close, 10,430 rows, ETPs and dead-book symbol-days already excluded. No engineered columns yet — that's what this notebook builds, per the handling rules in `1) data_exploration_report.md`.

In [1]:
import numpy as np
import pandas as pd

panel = pd.read_csv("data/panel_clean.csv", parse_dates=["date"])
panel.shape

(10430, 19)

## Base features (last snapshot)

`panel_clean.csv` already has the last pre-close snapshot per symbol-day and the target (`close_price`). "
"First pass: derive the level features from that single row — imbalance direction, spread, size normalized "
"by ADV, and the near/far and open-anchored moves.

In [2]:
panel["signed_shares"] = np.where(panel["side"] == "BUY", panel["shares"],
                            np.where(panel["side"] == "SELL", -panel["shares"], 0.0))
panel["signed_imbalance_ratio"] = panel["signed_shares"] / (panel["paired_shares"] + panel["shares"])

panel["spread"] = panel["ask"] - panel["bid"]
panel["spread_bps"] = panel["spread"] / panel["ref_price"] * 1e4
panel["urgency"] = panel["signed_imbalance_ratio"] * panel["spread_bps"]

panel["shares_over_adv"] = panel["shares"] / (panel["adv"] * 1000)
panel["paired_shares_over_adv"] = panel["paired_shares"] / (panel["adv"] * 1000)

panel["far_price_available"] = panel["far_price"] != 0
panel["near_far_spread_bps"] = np.where(
    panel["far_price_available"], (panel["near_price"] / panel["far_price"] - 1) * 1e4, np.nan
)  # far_price == 0.0 means no standalone closing-auction-only orders existed for this name -- a
   # real absence (mostly ETFs/ETNs), not bad data; explicit NaN + flag rather than a silent inf
panel["intraday_move_bps"] = (panel["ref_price"] / panel["open"] - 1) * 1e4

panel[["signed_imbalance_ratio", "spread_bps", "urgency", "shares_over_adv", "near_far_spread_bps", "intraday_move_bps"]].describe().T

,count,mean,std,min,25%,50%,75%,max
signed_imbalance_ratio,10430.0,0.001513,0.058714,-0.994444,0.000000,0.000000,0.003489,1.000000
spread_bps,10430.0,8.250664,11.810362,0.156050,2.896032,5.509106,9.895916,330.578512
urgency,10430.0,-0.012864,1.812696,-69.586067,0.000000,0.000000,0.012531,43.585278
shares_over_adv,10430.0,0.002343,0.007111,0.000000,0.000000,0.000000,0.002121,0.240281
near_far_spread_bps,10398.0,0.578463,190.259505,-6813.333333,0.000000,0.000000,0.000000,11039.396479
intraday_move_bps,10430.0,-13.763523,349.106798,-4056.521739,-143.444428,-1.292432,125.283791,4678.030303


## Time-series features (momentum, lags, volatility)

`panel_clean.csv` is one row per symbol-day, so it has no time-series information by itself — momentum, "
"lagged returns, and rolling volatility (the most consistently useful features in the literature) all need "
"multiple pre-close snapshots. Reloading a narrow slice of the raw feed — just the last 90 seconds before "
"close, restricted to the symbol-days already validated in `01)` — to compute those, rather than expanding "
"the panel to multiple rows per symbol-day. That keeps one feature vector per symbol-day, matching the "
"take-home's framing of a single entry decision, while still capturing the time-series signal the papers "
"found mattered most.

In [3]:
import tarfile

CLOSE = "16:00:00"

def load_all(tar_path):
    frames = []
    with tarfile.open(tar_path) as tf:
        for member in tf.getmembers():
            if not member.name.endswith(".csv.gz"):
                continue
            date_str = member.name.split(".")[0]
            d = pd.read_csv(tf.extractfile(member), compression="gzip")
            d = d.drop(columns=["ts.1"])
            d["date"] = pd.to_datetime(date_str, format="%Y%m%d")
            frames.append(d)
    df = pd.concat(frames, ignore_index=True)
    df["ts"] = pd.to_datetime(df["ts"])
    df["local_clock"] = df["local_time"].str.slice(11, 19)
    return df

raw = load_all("data/202503_imbalance.tar")

valid_keys = panel[["symbol", "date"]]
raw = raw.merge(valid_keys, on=["symbol", "date"], how="inner")
raw = raw[(raw["local_clock"] >= "15:58:00") & (raw["local_clock"] < CLOSE)].copy()
raw["signed_shares"] = np.where(raw["side"] == "BUY", raw["shares"],
                          np.where(raw["side"] == "SELL", -raw["shares"], 0.0))
raw["imbalance_ratio"] = raw["shares"] / (raw["paired_shares"] + raw["shares"])
raw["signed_imbalance_ratio"] = raw["signed_shares"] / (raw["paired_shares"] + raw["shares"])
raw = raw.sort_values("ts")

print(f"raw slice: {raw.shape[0]} rows across {raw[['symbol','date']].drop_duplicates().shape[0]} symbol-days")

raw slice: 1251600 rows across 10430 symbol-days


In [4]:
t0 = panel[["symbol", "date", "ts"]].rename(columns={"ts": "t0_ts"}).copy()
t0["t0_ts"] = pd.to_datetime(t0["t0_ts"])

lag_cols = ["ref_price", "imbalance_ratio", "signed_imbalance_ratio"]

for lag_s in [10, 30, 60]:
    targets = t0.copy()
    targets["ts"] = targets["t0_ts"] - pd.Timedelta(seconds=lag_s)
    targets = targets.sort_values("ts")

    lagged = pd.merge_asof(
        targets, raw[["ts", "symbol", "date"] + lag_cols],
        on="ts", by=["symbol", "date"], direction="backward",
    )
    lagged = lagged.rename(columns={c: f"{c}_lag{lag_s}s" for c in lag_cols})
    panel = panel.merge(lagged[["symbol", "date"] + [f"{c}_lag{lag_s}s" for c in lag_cols]],
                         on=["symbol", "date"], how="left")

    panel[f"imbalance_ratio_chg_{lag_s}s"] = panel["imbalance_ratio"] - panel[f"imbalance_ratio_lag{lag_s}s"]
    panel[f"signed_imbalance_ratio_chg_{lag_s}s"] = (
        panel["signed_imbalance_ratio"] - panel[f"signed_imbalance_ratio_lag{lag_s}s"]
    )
    panel[f"ref_price_ret_{lag_s}s_bps"] = (
        panel["ref_price"] / panel[f"ref_price_lag{lag_s}s"] - 1
    ) * 1e4

print("null rate in the 60s lag columns (should be near 0 -- only symbol-days with <60s of pre-close data would miss):")
print(panel[["ref_price_lag60s", "imbalance_ratio_lag60s"]].isna().mean())

null rate in the 60s lag columns (should be near 0 -- only symbol-days with <60s of pre-close data would miss):
ref_price_lag60s          0.0
imbalance_ratio_lag60s    0.0
dtype: float64


### Rolling volatility (last 60 seconds)

In [5]:
window = raw.merge(t0, on=["symbol", "date"], how="inner")
window = window[window["ts"] >= (window["t0_ts"] - pd.Timedelta(seconds=60))]

roll = window.groupby(["symbol", "date"]).agg(
    ref_price_std_60s=("ref_price", "std"),
    imbalance_ratio_std_60s=("imbalance_ratio", "std"),
    n_snapshots_60s=("ts", "count"),
).reset_index()

panel = panel.merge(roll, on=["symbol", "date"], how="left")
panel["ref_price_std_60s_bps"] = panel["ref_price_std_60s"] / panel["ref_price"] * 1e4

print(panel[["ref_price_std_60s_bps", "imbalance_ratio_std_60s", "n_snapshots_60s"]].describe().T)

                           count       mean       std   min        25%  \
ref_price_std_60s_bps    10430.0   3.973041  5.403060   0.0   1.821346   
imbalance_ratio_std_60s  10430.0   0.018783  0.022752   0.0   0.004164   
n_snapshots_60s          10430.0  60.395686  0.489021  60.0  60.000000   

                               50%        75%         max  
ref_price_std_60s_bps     2.891879   4.541696  283.020441  
imbalance_ratio_std_60s   0.013044   0.024716    0.377123  
n_snapshots_60s          60.000000  61.000000   61.000000  


## Assemble and save

In [6]:
drop_cols = [c for c in panel.columns if c.startswith("ref_price_lag") or c.startswith("imbalance_ratio_lag")
             or c.startswith("signed_imbalance_ratio_lag") or c == "ref_price_std_60s"]
features = panel.drop(columns=drop_cols)

print(f"features: {features.shape[0]} rows, {features.shape[1]} columns")
print(f"any nulls in engineered columns: {features.filter(regex='chg_|ret_|std_|urgency|spread|adv|move').isna().sum().sum()}")

features.to_csv("data/features.csv", index=False)
print("wrote data/features.csv")

features: 10430 rows, 41 columns
any nulls in engineered columns: 32


wrote data/features.csv


**41 columns, 10,430 rows, one row per symbol-day — 22 raw/base + 19 engineered.**

- Level (last snapshot): `signed_imbalance_ratio`, `spread_bps`, `urgency` (spread × imbalance interaction),
  `shares_over_adv`, `paired_shares_over_adv`, `near_far_spread_bps` (+ `far_price_available` flag),
  `intraday_move_bps`.
- Time-series (10s/30s/60s lookback, computed from a raw reload of the last 90s of the pre-close window,
  merged back via `merge_asof`): imbalance change, signed-imbalance change, price return in bps.
- Volatility: `ref_price_std_60s_bps`, `imbalance_ratio_std_60s`, `n_snapshots_60s` (coverage check — should be
  ~60-61 for everyone; worth checking before trusting the volatility numbers on any row where it isn't).

Deliberately not done here, per handling rule 8 from the EDA: no scaling/standardization, no train/test split
— those need to happen after a split so no statistic is computed over data a model wouldn't have had at
decision time. That's the modeling notebook's job, not this one's.